In [7]:
import pandas as pd
import numpy as np
import os
import warnings

In [8]:
RAW_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

print("Raw data path      :", RAW_PATH)
print("Processed data path:", PROCESSED_PATH)

Raw data path      : ../data/raw
Processed data path: ../data/processed


In [9]:
files = os.listdir(RAW_PATH)

print("Available files:\n")

for file in files:
    print(" -", file)

Available files:

 - City.xlsx
 - Continent.xlsx
 - Country.xlsx
 - Item.xlsx
 - Mode.xlsx
 - Region.xlsx
 - Transaction.xlsx
 - Type.xlsx
 - Updated_Item.xlsx
 - User.xlsx


In [10]:
cities = pd.read_excel(
    os.path.join(RAW_PATH, "City.xlsx")
)

print("Cities loaded successfully.")
print("Shape:", cities.shape)

display(cities.head())

Cities loaded successfully.
Shape: (9143, 3)


,CityId,CityName,CountryId
0,0,-,0
1,1,Douala,1
2,2,South Region,1
3,3,N'Djamena,2
4,4,Kigali,3


In [11]:
excel_files = [
    file for file in os.listdir(RAW_PATH)
    if file.lower().endswith((".xlsx", ".xls"))
]

print("Excel files found:")
for file in excel_files:
    print(" -", file)

Excel files found:
 - City.xlsx
 - Continent.xlsx
 - Country.xlsx
 - Item.xlsx
 - Mode.xlsx
 - Region.xlsx
 - Transaction.xlsx
 - Type.xlsx
 - Updated_Item.xlsx
 - User.xlsx


In [13]:
for file in excel_files:
    file_path = os.path.join(RAW_PATH, file)

    try:
        excel_file = pd.ExcelFile(file_path)

        print(f"\n{file}")
        print("Sheets:", excel_file.sheet_names)

    except Exception as e:
        print(f"Could not read {file}: {e}")


City.xlsx
Sheets: ['Cities']

Continent.xlsx
Sheets: ['Continents']

Country.xlsx
Sheets: ['Countries']

Item.xlsx
Sheets: ['Item']

Mode.xlsx
Sheets: ['VisitingMode']

Region.xlsx
Sheets: ['Regions']

Transaction.xlsx
Sheets: ['Transaction']

Type.xlsx
Sheets: ['Types']

Updated_Item.xlsx
Sheets: ['Sheet1']

User.xlsx
Sheets: ['User']


In [14]:
datasets = {}

print("Dataset dictionary initialized.")

Dataset dictionary initialized.


In [15]:
datasets["cities"] = cities.copy()

print("Datasets currently loaded:")
print(list(datasets.keys()))

Datasets currently loaded:
['cities']


In [16]:
def standardize_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.lower()
    )

    return df

In [17]:
cities_std = standardize_columns(cities)

print(cities_std.columns.tolist())

['cityid', 'cityname', 'countryid']


In [18]:
cities_std = cities_std.rename(columns={
    "cityid": "city_id",
    "cityname": "city_name",
    "countryid": "country_id"
})

display(cities_std.head())

,city_id,city_name,country_id
0,0,-,0
1,1,Douala,1
2,2,South Region,1
3,3,N'Djamena,2
4,4,Kigali,3


In [19]:
print("Total records :", len(cities_std))
print("Unique city_id:", cities_std["city_id"].nunique())
print("Missing city_id:", cities_std["city_id"].isna().sum())
print("Duplicate city_id:", cities_std["city_id"].duplicated().sum())

Total records : 9143
Unique city_id: 9143
Missing city_id: 0
Duplicate city_id: 0


In [20]:
relationships = {
    "transaction_user": {
        "left_key": "user_id",
        "right_key": "user_id"
    },

    "user_city": {
        "left_key": "city_id",
        "right_key": "city_id"
    },

    "city_country": {
        "left_key": "country_id",
        "right_key": "country_id"
    },

    "country_region": {
        "left_key": "region_id",
        "right_key": "region_id"
    },

    "region_continent": {
        "left_key": "continent_id",
        "right_key": "continent_id"
    },

    "transaction_attraction": {
        "left_key": "attraction_id",
        "right_key": "attraction_id"
    },

    "attraction_type": {
        "left_key": "attraction_type_id",
        "right_key": "attraction_type_id"
    }
}

relationships

{'transaction_user': {'left_key': 'user_id', 'right_key': 'user_id'},
 'user_city': {'left_key': 'city_id', 'right_key': 'city_id'},
 'city_country': {'left_key': 'country_id', 'right_key': 'country_id'},
 'country_region': {'left_key': 'region_id', 'right_key': 'region_id'},
 'region_continent': {'left_key': 'continent_id', 'right_key': 'continent_id'},
 'transaction_attraction': {'left_key': 'attraction_id',
  'right_key': 'attraction_id'},
 'attraction_type': {'left_key': 'attraction_type_id',
  'right_key': 'attraction_type_id'}}

In [21]:
def validate_join(left_df, right_df, left_key, right_key):
    """
    Validate key relationships before merging datasets.
    """

    print("=" * 60)
    print("JOIN VALIDATION")
    print("=" * 60)

    print(f"Left key : {left_key}")
    print(f"Right key: {right_key}")

    print("\nLeft dataset:")
    print("Rows:", len(left_df))
    print("Unique keys:", left_df[left_key].nunique())
    print("Missing keys:", left_df[left_key].isna().sum())

    print("\nRight dataset:")
    print("Rows:", len(right_df))
    print("Unique keys:", right_df[right_key].nunique())
    print("Missing keys:", right_df[right_key].isna().sum())

    unmatched = (
        ~left_df[left_key]
        .isin(right_df[right_key])
    )

    print("\nUnmatched left records:", unmatched.sum())

    return unmatched

In [23]:
city_key_check = cities_std[
    cities_std["city_id"].notna()
]

print("City records available:", len(city_key_check))
print("Unique city IDs:", city_key_check["city_id"].nunique())
print("Duplicate city IDs:", city_key_check["city_id"].duplicated)

City records available: 9143
Unique city IDs: 9143
Duplicate city IDs: <bound method Series.duplicated of 0          0
1          1
2          2
3          3
4          4
        ... 
9138    9138
9139    9139
9140    9140
9141    9141
9142    9142
Name: city_id, Length: 9143, dtype: int64>


In [24]:
unresolved_city = cities_std[
    cities_std["city_name"].isna()
]

print("Unresolved city records:")
display(unresolved_city)

Unresolved city records:


,city_id,city_name,country_id
6879,6879,NaN,151


In [25]:
city_data_dictionary = pd.DataFrame({
    "Column": [
        "city_id",
        "city_name",
        "country_id"
    ],
    "Description": [
        "Unique identifier for each city",
        "Name of the city",
        "Identifier of the country associated with the city"
    ],
    "Role": [
        "Primary Key",
        "Descriptive Feature",
        "Foreign Key"
    ],
    "Data_Type": [
        str(cities_std["city_id"].dtype),
        str(cities_std["city_name"].dtype),
        str(cities_std["country_id"].dtype)
    ]
})

display(city_data_dictionary)

,Column,Description,Role,Data_Type
0,city_id,Unique identifier for each city,Primary Key,int64
1,city_name,Name of the city,Descriptive Feature,object
2,country_id,Identifier of the country associated with the ...,Foreign Key,int64


In [26]:
city_output_path = os.path.join(
    PROCESSED_PATH,
    "cities_standardized.csv"
)

cities_std.to_csv(
    city_output_path,
    index=False
)

print("Standardized Cities dataset saved:")
print(city_output_path)

Standardized Cities dataset saved:
../data/processed\cities_standardized.csv
